# Decision Gate (Task 4)
Profiles `profile_gpu_mpc.py` + `profile_gpu_ppo.py` on Colab T4 vs local CPU baseline.
Produces `DECISION.md`: GPU-justified iff throughput >= 5x CPU AND SM util mean >= 50%.
Before running: set runtime -> GPU T4. Expect ~10-15 min total.

In [ ]:
# Cell 1: unzip bundle (upload decision_gate_bundle.zip via Files panel first)
import os
assert os.path.exists('/content/decision_gate_bundle.zip'), 'Upload decision_gate_bundle.zip to /content first'
!unzip -q -o /content/decision_gate_bundle.zip -d /content
!ls /content/src

In [ ]:
# Cell 2: install Colab-specific deps
!pip install -q -r /content/requirements_colab.txt

In [ ]:
# Cell 3: env vars + GPU sanity
import os, subprocess
os.environ['TEST_SRC_ROOT'] = '/content/src/test'
os.environ['ADK_SRC_ROOT']  = '/content/src/adk'
print(subprocess.check_output(['nvidia-smi'], text=True)[:400])

In [ ]:
# Cell 4: generate 1 dataset (plain_small_cons seed 0)
%cd /content/src/benchmark
!python -m generator.generate --preset presets/plain_small_cons.yaml --seed 0 --out /content/data
!ls /content/data

In [ ]:
# Cell 5: MPC profile
%cd /content/src/benchmark
!python profiling/profile_gpu_mpc.py /content/data

In [ ]:
# Cell 6: PPO profile (25k timesteps)
%cd /content/src/benchmark
!python profiling/profile_gpu_ppo.py /content/data

In [ ]:
# Cell 7: build DECISION.md from CPU + T4 profiles
import json, pathlib
bench = pathlib.Path('/content/src/benchmark')
cpu = json.loads((bench / 'profiling/cpu_profile.json').read_text())
mpc = json.loads((bench / 'profiling/gpu_mpc_profile.json').read_text())
ppo = json.loads((bench / 'profiling/gpu_ppo_profile.json').read_text())
GPU_UTIL_BAR = 50.0
THROUGHPUT_BAR = 5.0
mpc_util = float(mpc.get('gpu_sm_util_mean_pct') or 0)
ppo_util = float(ppo.get('gpu_sm_util_mean_pct') or 0)
mpc_cpu_sps = 1.0 / cpu['mpc_cpu_s_per_step_plain_small']
mpc_gpu_sps = 1.0 / mpc['s_per_step']
mpc_ratio = mpc_gpu_sps / mpc_cpu_sps
ppo_cpu_sps = cpu['ppo_cpu_steps_per_s_plain_small']
ppo_gpu_sps = ppo['steps_per_s']
ppo_ratio = ppo_gpu_sps / ppo_cpu_sps
mpc_ok_util = mpc_util >= GPU_UTIL_BAR
mpc_ok_tput = mpc_ratio >= THROUGHPUT_BAR
ppo_ok_util = ppo_util >= GPU_UTIL_BAR
ppo_ok_tput = ppo_ratio >= THROUGHPUT_BAR
mpc_dec = 'GPU' if (mpc_ok_util and mpc_ok_tput) else 'CPU'
ppo_dec = 'GPU' if (ppo_ok_util and ppo_ok_tput) else 'CPU'
lines = []
lines.append('# Decision Gate Result')
lines.append('')
lines.append('Source: Colab T4, plain_small_cons seed 0 (800 blocks).')
lines.append(f'Rule: GPU when throughput >= {THROUGHPUT_BAR}x CPU AND SM util mean >= {GPU_UTIL_BAR}%.')
lines.append('')
lines.append('## MPC')
lines.append(f"- CPU baseline    : {mpc_cpu_sps:.3f} steps/s ({cpu['mpc_cpu_s_per_step_plain_small']:.3f} s/step)")
lines.append(f"- GPU T4          : {mpc_gpu_sps:.3f} steps/s ({mpc['s_per_step']:.3f} s/step)")
lines.append(f"- Throughput gain : {mpc_ratio:.2f}x  (bar {THROUGHPUT_BAR}x) {'PASS' if mpc_ok_tput else 'FAIL'}")
lines.append(f"- SM util mean    : {mpc_util:.1f}%    (bar {GPU_UTIL_BAR}%) {'PASS' if mpc_ok_util else 'FAIL'}")
lines.append(f"- Decision        : {mpc_dec}")
lines.append('')
lines.append('## PPO')
lines.append(f"- CPU baseline    : {ppo_cpu_sps:.3f} steps/s ({cpu['ppo_cpu_total_timesteps']} ts smoke)")
lines.append(f"- GPU T4          : {ppo_gpu_sps:.3f} steps/s ({ppo['total_timesteps']} ts, {ppo['wall_s']:.1f} s)")
lines.append(f"- Throughput gain : {ppo_ratio:.2f}x  (bar {THROUGHPUT_BAR}x) {'PASS' if ppo_ok_tput else 'FAIL'}")
lines.append(f"- SM util mean    : {ppo_util:.1f}%    (bar {GPU_UTIL_BAR}%) {'PASS' if ppo_ok_util else 'FAIL'}")
lines.append(f"- Decision        : {ppo_dec}")
lines.append('')
lines.append('## CPU reference (cpu_profile.json)')
lines.append(f"- generator       : {cpu['generator_plain_small_s']:.1f} s")
lines.append(f"- random          : {cpu['random_plain_small_s']:.3f} s")
lines.append(f"- greedy          : {cpu['greedy_plain_small_s']:.1f} s")
lines.append(f"- ga-500gen (est) : {cpu['ga_plain_small_500gen_estimate_s']:.1f} s")
decision_md = '\n'.join(lines)
(bench / 'profiling/DECISION.md').write_text(decision_md, encoding='utf-8')
print('=== gpu_mpc_profile.json ==='); print(json.dumps(mpc, indent=2))
print('=== gpu_ppo_profile.json ==='); print(json.dumps(ppo, indent=2))
print('=== DECISION.md ===\n' + decision_md)

In [ ]:
# Cell 8: package artifacts for download
!cd /content/src/benchmark/profiling && zip -j /content/decision_artifacts.zip gpu_mpc_profile.json gpu_ppo_profile.json DECISION.md
print('Download /content/decision_artifacts.zip via Files panel')